In [37]:
import torch 
from torch import nn
from torch.nn import functional as F

In [38]:
data = [[[1, 2, 3],[4, 5, 6],[7, 8, 9]]]
my_tensor = torch.tensor(data)
print(my_tensor)

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])


In [39]:
shape = (2,3)

ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.rand(shape)
print(f"Random tensor: \n{random}\n")
print(f"Ones tensor: \n{ones}\n")
print(f"Zeros tensor: \n{zeros}\n")

print(shape)

Random tensor: 
tensor([[0.1716, 0.3336, 0.5782],
        [0.0600, 0.2846, 0.2007]])

Ones tensor: 
tensor([[1., 1., 1.],
        [1., 1., 1.]])

Zeros tensor: 
tensor([[0., 0., 0.],
        [0., 0., 0.]])

(2, 3)


In [40]:
#mimicking like given a tensor of shape (2,3) we need another tensor of shape same .

template = torch.tensor([[1,2], [3,4]]) 

#random_tensor = torch.rand_like(template)
random_tensor = torch.randn_like(template , dtype = torch.float32)
print(f"Random tensor :  \n {random_tensor}\n")

Random tensor :  
 tensor([[-0.0072,  1.5980],
        [ 0.1115, -0.0392]])



In [41]:
# Shape  , type and device 
## float32 are important for neural networks as they are used to calculate the gradients and update the weights of the model during training for learning .

tensor = torch.tensor([[2, 3]] , dtype = torch.float32 )

print(f"Shape of tensor : {tensor.shape}")
print(f"Datatype of tensor : {tensor.dtype}")
print(f"Device tensor is stored on : {tensor.device}")

Shape of tensor : torch.Size([1, 2])
Datatype of tensor : torch.float32
Device tensor is stored on : cpu


#### Autograd = pytorch automatic deferentiation engine that powers neural network training . It records all the operations that happen on a tensor and allows us to calculate gradients automatically when we call .backward() on a tensor .

In [42]:
x = torch.tensor([[[1, 2, 3], [4, 5, 6]]])
w = torch.tensor([[[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]] , requires_grad = True)

In [43]:
print(f"X : \n {x.requires_grad}\n")
print(f"W : \n {w.requires_grad}\n")

X : 
 False

W : 
 True



In [44]:
a = torch.tensor(2.0 , requires_grad = True)
b = torch.tensor(3.0 , requires_grad = True)
x = torch.tensor(4.0 , requires_grad= True )

In [45]:
y = a + b 
z = y * x

print(f"z : {z} \n")
print(f"z : {z.grad_fn} \n")
print(f"y grad fn : {y.grad_fn} \n")
print(f"x grad fn : {x.grad_fn} \n")

z : 20.0 

z : <MulBackward0 object at 0x0000024E9DF2A650> 

y grad fn : <AddBackward0 object at 0x0000024E9DF2A650> 

x grad fn : None 



In [46]:
a = torch.tensor(  [[1,2] , [3,4] ] ) 
b = torch.tensor( [[5,6] , [7,8]] )
c = a* b 
print(f"c : {c} \n")
# element wise multiplication  [[5,12] , [21,32]]   




c : tensor([[ 5, 12],
        [21, 32]]) 



In [47]:
# Matrix multiplication
a = torch.tensor(  [[1,2] , [3,4] ] )
b = torch.tensor( [[5,6] , [7,8]] )
c = a @ b
print(f"c : {c} \n")

c : tensor([[19, 22],
        [43, 50]]) 



In [48]:
x = torch.arange(1, 10)
x = x.view(1, 3, 3)
print(f"x : \n {x}\n")

x : 
 tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])



In [49]:
data = torch.tensor([
    [10, 11, 12, 13],  # row 0
    [20, 21, 22, 23],  # row 1
    [30, 31, 32, 33]   # row 2
])

# Our "personalized list" of column indices to select from each row
indices_to_select = torch.tensor([[2], [0], [3]])

# Gather from `data` along dim=1 (the column dimension)
selected_values = torch.gather(data, dim=1, index=indices_to_select)

print(f"Selected Values:\n {selected_values}")

Selected Values:
 tensor([[12],
        [20],
        [33]])


## Implementing a Linear Regression

ŷ = XW + b

Where:

X is our input data.
W is the weight parameter.
b is the bias parameter.
ŷ is our model's prediction.

goal is to find the best W and b to make ŷ as close to the true y as possible.

In [50]:
"""
Linear Regression from Scratch in PyTorch (No nn.Module)
Target function: y = 2x + 1
"""

import torch

# ──────────────────────────────────────────────
# 1. CREATE TOY DATA  (y = 2x + 1 + noise)
# ──────────────────────────────────────────────
torch.manual_seed(42)

X = torch.linspace(0, 5, 50).unsqueeze(1)          # shape: (50, 1)
y_true = 2 * X + 1 + 0.3 * torch.randn_like(X)    # noisy targets

# ──────────────────────────────────────────────
# 2. INIT LEARNABLE PARAMETERS
# ──────────────────────────────────────────────
# These are the two numbers the model will learn:
#   w (weight) → should converge to ~2
#   b (bias)   → should converge to ~1

w = torch.randn(1, requires_grad=True)   # random start
b = torch.zeros(1, requires_grad=True)   # start at 0

# ──────────────────────────────────────────────
# 3. HYPERPARAMETERS
# ──────────────────────────────────────────────
lr = 0.01        # learning rate – how big a step we take
epochs = 200     # how many times we loop over the data

# ──────────────────────────────────────────────
# 4. TRAINING LOOP
# ──────────────────────────────────────────────
for epoch in range(epochs):

    # --- Forward pass: predict y ---
    y_pred = w * X + b                          # our linear model

    # --- Compute loss (Mean Squared Error) ---
    loss = ((y_pred - y_true) ** 2).mean()      # scalar

    # --- Backward pass: compute gradients ---
    loss.backward()          # fills w.grad and b.grad

    # --- Update parameters (gradient descent) ---
    with torch.no_grad():                       # don't track these ops
        w -= lr * w.grad
        b -= lr * b.grad

    # --- Zero gradients for next iteration ---
    w.grad.zero_()
    b.grad.zero_()

    # --- Print progress every 20 epochs ---
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>3d} | Loss: {loss.item():.4f} | "
              f"w = {w.item():.4f}, b = {b.item():.4f}")

# ──────────────────────────────────────────────
# 5. FINAL RESULT
# ──────────────────────────────────────────────
print("\n--- Training Complete ---")
print(f"Learned:  y = {w.item():.4f}·x + {b.item():.4f}")
print(f"Actual:   y = 2.0000·x + 1.0000")

Epoch  20 | Loss: 0.1397 | w = 2.0157, b = 0.8123
Epoch  40 | Loss: 0.1064 | w = 2.0547, b = 0.8449
Epoch  60 | Loss: 0.1048 | w = 2.0502, b = 0.8623
Epoch  80 | Loss: 0.1035 | w = 2.0454, b = 0.8779
Epoch 100 | Loss: 0.1023 | w = 2.0411, b = 0.8921
Epoch 120 | Loss: 0.1014 | w = 2.0372, b = 0.9051
Epoch 140 | Loss: 0.1007 | w = 2.0336, b = 0.9168
Epoch 160 | Loss: 0.1000 | w = 2.0303, b = 0.9275
Epoch 180 | Loss: 0.0995 | w = 2.0273, b = 0.9373
Epoch 200 | Loss: 0.0991 | w = 2.0246, b = 0.9461

--- Training Complete ---
Learned:  y = 2.0246·x + 0.9461
Actual:   y = 2.0000·x + 1.0000


In [51]:
# We'll create a "batch" of 10 data points
N = 10
# Each data point has 1 feature
D_in = 1
# The output for each data point is a single value
D_out = 1

# Create our input data X
# Shape: (10 rows, 1 column)
X = torch.randn(N, D_in)

# Create our true target labels y by applying the "true" function
# and adding some noise for realism
true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1 # Add a little noise

print(f"Input Data X (first 3 rows):\n {X[:3]}\n")
print(f"True Labels y_true (first 3 rows):\n {y_true[:3]}")

Input Data X (first 3 rows):
 tensor([[ 1.2580],
        [-1.5890],
        [-1.1208]])

True Labels y_true (first 3 rows):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])


In [52]:
# Initialize our parameters with random values
# Shapes must be correct for matrix multiplication: X(10,1) @ W(1,1) -> (10,1)
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Initial Weight W:\n {W}\n")
print(f"Initial Bias b:\n {b}")

Initial Weight W:
 tensor([[-0.7290]], requires_grad=True)

Initial Bias b:
 tensor([0.7277], requires_grad=True)


In [53]:
# Perform the forward pass to get our first prediction
y_hat = X @ W + b

print(f"Shape of our prediction y_hat: {y_hat.shape}\n")
print(f"Prediction y_hat (first 3 rows):\n {y_hat[:3]}\n")
print(f"True Labels y_true (first 3 rows):\n {y_true[:3]}")

Shape of our prediction y_hat: torch.Size([10, 1])

Prediction y_hat (first 3 rows):
 tensor([[-0.1894],
        [ 1.8860],
        [ 1.5447]], grad_fn=<SliceBackward0>)

True Labels y_true (first 3 rows):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])


In [54]:
# y_hat is our prediction from the forward pass
# y_true is the ground truth
# Let's calculate the loss manually
error = y_hat - y_true
squared_error = error ** 2
loss = squared_error.mean()

print(f"Prediction (first 3):\n {y_hat[:3]}\n")
print(f"Truth (first 3):\n {y_true[:3]}\n")
print(f"Loss (a single number): {loss}")

Prediction (first 3):
 tensor([[-0.1894],
        [ 1.8860],
        [ 1.5447]], grad_fn=<SliceBackward0>)

Truth (first 3):
 tensor([[ 3.7902],
        [-2.1222],
        [-1.3228]])

Loss (a single number): 10.136190414428711
